# EYEWAZ — train your Urdu Piper voice

**Before running:** Runtime → Change runtime type → **GPU (T4)**. Upload your
`dataset-<VOICE>.zip` via the Files panel (📁). Then Runtime → **Run all**.

⚠️ Cell 2 switches the runtime to **Python 3.10** (Piper needs it) and **restarts**
the kernel — that's expected. When it restarts, just click **Run all again**; it
skips the restart the second time and continues.


### 1. Check the GPU is on


In [ ]:
!nvidia-smi -L || print('NO GPU — set Runtime > Change runtime type > GPU first!')


### 2. Get Python 3.10 (piper-phonemize has no wheels for Colab's 3.12)
Restarts the kernel the first time. Re-run **Run all** after it restarts.


In [ ]:
import sys
if sys.version_info[:2] != (3, 10):
    !pip install -q condacolab
    import condacolab; condacolab.install()   # installs Python 3.10 base, RESTARTS here
else:
    print('Python 3.10 ready:', sys.version.split()[0])


### 3. Install Piper training (now on Python 3.10)


In [ ]:
!python --version
import os
if not os.path.exists('/content/piper'):
    !git clone -q https://github.com/rhasspy/piper /content/piper
%cd /content/piper/src/python
!pip install -q -e .
!pip install -q -r requirements.txt
!bash build_monotonic_align.sh
print('Piper training installed.')


### 4. Settings + unzip your dataset
Upload `dataset-<VOICE>.zip` to `/content/` first (Files panel).


In [ ]:
import os
VOICE = 'male'          # must match your uploaded dataset-<VOICE>.zip
DATA  = f'/content/dataset-{VOICE}'
OUT   = f'/content/train-{VOICE}'
zip_path = f'/content/dataset-{VOICE}.zip'
# The Python-3.10 switch in cell 2 restarts the runtime and clears /content, so
# re-upload here if the zip is gone (an upload button appears).
if not os.path.exists(zip_path):
    from google.colab import files
    print(f'Select dataset-{VOICE}.zip to upload:')
    up = files.upload()
    for name in up:
        if name != os.path.basename(zip_path):
            os.replace(f'/content/{name}', zip_path)
assert os.path.exists(zip_path), f'dataset-{VOICE}.zip not uploaded'
!unzip -oq {zip_path} -d /content/
print('clips:', len(os.listdir(f'{DATA}/wav')))


### 5. Preprocess (Urdu phonemes via espeak-ng)


In [ ]:
!python -m piper_train.preprocess \
  --language ur \
  --input-dir {DATA} \
  --output-dir {OUT} \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050


### 6. Pretrained checkpoint to fine-tune from
Lessac (en_US) medium — acoustic model transfers; Urdu phonemes come from espeak-ng.


In [ ]:
# If this 404s, browse https://huggingface.co/datasets/rhasspy/piper-checkpoints and grab any */medium/*.ckpt
URL='https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
!wget -q --show-progress -O /content/pretrained.ckpt "$URL"
import os; print('checkpoint MB:', round(os.path.getsize('/content/pretrained.ckpt')/1e6,1))


### 7. Train
Watch the audio samples under the ▶ outputs. ~6 min of data = a **pilot** (rough).
**Stop the cell** (■) when samples sound good enough, then run the next cell.


In [ ]:
!python -m piper_train \
  --dataset-dir {OUT} \
  --accelerator gpu --devices 1 \
  --batch-size 16 \
  --max_epochs 4000 \
  --checkpoint-epochs 5 \
  --quality medium \
  --resume_from_checkpoint /content/pretrained.ckpt


### 8. Export to ONNX + download


In [ ]:
ck = !ls -t {OUT}/lightning_logs/version_0/checkpoints/*.ckpt | head -1
ck = ck[0]; print('exporting', ck)
!python -m piper_train.export_onnx {ck} /content/eyewaz-urdu-{VOICE}.onnx
!cp {OUT}/config.json /content/eyewaz-urdu-{VOICE}.onnx.json
from google.colab import files
files.download(f'/content/eyewaz-urdu-{VOICE}.onnx')
files.download(f'/content/eyewaz-urdu-{VOICE}.onnx.json')


### Done
Send me `eyewaz-urdu-<VOICE>.onnx` + `.onnx.json` and I'll wire the voice into every EYEWAZ surface.
